# COP/USD Intelligence — Exploration Notebook

This notebook is for interactive exploration of FX data, model tuning, and visualisation.
Run `make install` before starting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True

## 1. Fetch historical COP/USD data

In [ ]:
from cop_fx.data.fx_fetcher import FXFetcher

fetcher = FXFetcher()
df = fetcher.fetch(lookback_days=365)
df.tail()

In [ ]:
fig, ax = plt.subplots()
ax.plot(df['ds'], df['y'], linewidth=1.2, color='#1f77b4')
ax.set_title('COP / USD — Daily Rate (last 365 days)')
ax.set_xlabel('Date')
ax.set_ylabel('COP per 1 USD')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. Prophet forecast

In [ ]:
from cop_fx.timeseries.models import ProphetForecaster

train = df.iloc[:-10]
test  = df.iloc[-10:]

prophet = ProphetForecaster()
result  = prophet.fit_predict(train, horizon_days=10)
result.forecast

In [ ]:
fig, ax = plt.subplots()
ax.plot(train['ds'], train['y'], label='Train', linewidth=1)
ax.plot(test['ds'],  test['y'],  label='Actual', color='green', linewidth=1.5)
ax.plot(result.forecast['ds'], result.forecast['yhat'], label='Prophet', color='orange', linewidth=1.5, linestyle='--')
ax.fill_between(
    result.forecast['ds'],
    result.forecast['yhat_lower'],
    result.forecast['yhat_upper'],
    alpha=0.2, color='orange', label='95% CI'
)
ax.set_title('Prophet 10-day COP/USD Forecast')
ax.legend()
plt.tight_layout()
plt.show()

## 3. ARIMA forecast

In [ ]:
from cop_fx.timeseries.models import ARIMAForecaster

arima  = ARIMAForecaster()
result_arima = arima.fit_predict(train, horizon_days=10)
result_arima.forecast

## 4. Ensemble

In [ ]:
from cop_fx.timeseries.models import ensemble_forecast
from cop_fx.timeseries.evaluator import evaluate

ens = ensemble_forecast([result, result_arima])

for r in [result, result_arima]:
    metrics = evaluate(r, test)
    print(metrics)

## 5. News fetch & analysis

In [ ]:
from cop_fx.data.news_fetcher import NewsFetcher

news = NewsFetcher()
articles = news.fetch()
print(f'Fetched {len(articles)} articles')
for a in articles[:5]:
    print(f'  [{a.source}] {a.title}')

In [ ]:
from cop_fx.analysis.news_analyzer import NewsAnalyzer

analyzer = NewsAnalyzer()
analyzed = analyzer.analyze_batch(articles[:10])

for a in analyzed:
    icon = '🟢' if a.bullish_cop else '🔴'
    print(f'{icon} [{a.severity:6s}] {a.topic:20s} | {a.article.title[:70]}')

## 6. Full pipeline (dry run)

In [ ]:
from cop_fx.agents.graph import run_pipeline

state = run_pipeline(publish_enabled=False)
print(state['report_markdown'])